In [2]:
import math
import numpy as np
import pandas as pd

## 1. A continuación, se define el método de Chun de la siguiente manera:

$$
y_k = x_k - \frac{f(x_k)}{f'(x_k)}
$$

$$
x_{k+1} = y_k - \frac{f(x_k) + 2f(y_k)}{f'(x_k)}
$$

In [3]:
# a)  Programar el método de Chun en Python.
def ChunACOC(f, df, x0, tol, maxiter):
    iters = 0
    error = tol + 1
    lista_errores = []  # Usamos una lista limpia

    while error > tol and iters <= maxiter:
        # 1. Evaluar función y derivada en x0
        fxk = f(x0)
        dfxk = df(x0)
        
        # 2. Calcular yk (Predictor)
        y = x0 - fxk / dfxk
        fyk = f(y)
        
        # 3. Calcular x (Corrector - Fórmula de Chun)
        x = y - ((fxk + 2*fyk) / fxk) * (fyk / dfxk)
        
        # 4. Calcular error y guardar
        error = math.fabs(x - x0)
        lista_errores.append(error) # Agregamos el error a la lista
        iters = iters + 1
        x0 = x

    errores_np = np.array(lista_errores) 
    
    if len(errores_np) > 2:
        ACOC = np.log(errores_np[2:]/errores_np[1:-1]) / np.log(errores_np[1:-1]/errores_np[0:-2])
    else:
        ACOC = "Muy pocas iteraciones para calcular ACOC"
    
    return [x, iters, error, ACOC]

In [4]:
# b) Resolver la ecuación utilizando el método de Chun la ecuación 
f1 = lambda x: (x - 1)**3 - 1 + np.exp(x)
df1 = lambda x: 3*(x - 1)**2 + np.exp(x)

sol, iters, error, acoc = ChunACOC(f1, df1, 0.1, 1e-7, 100)

In [5]:
print(" MÉTODO DE CHUN ")
print("Solución aproximada: %s" % sol)
print("Cantidad de iteraciones: %s" % iters)
print("Error final: %s" % error)
print("ACOC por iteración: %s" % acoc)

 MÉTODO DE CHUN 
Solución aproximada: 0.2975706656987736
Cantidad de iteraciones: 3
Error final: 1.3433698597964394e-13
ACOC por iteración: [4.02460381]


## 3. El método MMC2 está definido de la siguiente manera:

$$
y_k = x_k - \frac{f(x_k)}{f'(x_k)}
$$

$$
z_k = y_k - \frac{f(x_k)}{f(x_k)-2f(y_k)}\frac{f(y_k)}{f'(x_k)}
$$

$$
x_{k+1} = z_k - \left(2 + 3\frac{f(y_k)}{f(x_k)} - \frac{f(x_k)}{f(x_k)-f(y_k)}\right)\frac{f(x_k)}{f'(x_k)}
$$

In [6]:
def MMC2_ACOC(f, df, x0, tol, maxiter):
    iters = 0
    error = tol + 1
    lista_errores = [] 
    while error > tol and iters <= maxiter:
        fxk = f(x0)
        dfxk = df(x0)
        
        yk = x0 - fxk / dfxk
        fyk = f(yk) 
        
        zk = yk - (fxk / (fxk - 2*fyk)) * (fyk / dfxk)
        fzk = f(zk) # Evaluamos f en z_k
        

        termino_izq = 2 + 3*(fyk / fxk) - (fxk / (fxk - fyk))
        termino_der = fzk / dfxk
        x_nuevo = zk - termino_izq * termino_der
        error = math.fabs(x_nuevo - x0)
        lista_errores.append(error)
        iters = iters + 1
        x0 = x_nuevo  # El nuevo punto pasa a ser el actual

    # --- Cálculo del ACOC (Fuera del bucle) ---
    errores_np = np.array(lista_errores)
    if len(errores_np) > 2:
        # Aplicamos la misma fórmula vectorizada
        ACOC = np.log(errores_np[2:]/errores_np[1:-1]) / np.log(errores_np[1:-1]/errores_np[0:-2])
    else:
        ACOC = "Muy pocas iteraciones"

    return [x_nuevo, iters, error, ACOC]

In [7]:
f3 = lambda x: np.sqrt(x**2 + 2*x + 5) - 2*np.sin(x) + x**2 - 3

# 2. Definimos la derivada f'(x) (la que calculamos a mano arriba)
df3 = lambda x: (2*x + 2)/(2*np.sqrt(x**2 + 2*x + 5)) - 2*np.cos(x) + 2*x

# 3. Llamamos al método con x0 = 3 y tolerancia 10^-12
sol, iters, error, acoc = MMC2_ACOC(f3, df3, 3.0, 1e-12, 100)


In [8]:
print(" MÉTODO DE MMC2 ")
print("Solución aproximada: %s" % sol)
print("Cantidad de iteraciones: %s" % iters)
print("Error final: %s" % error)
print("ACOC por iteración: %s" % acoc)

 MÉTODO DE MMC2 
Solución aproximada: 1.3641174263537554
Cantidad de iteraciones: 4
Error final: 1.5543122344752192e-15
ACOC por iteración: [4.4498866  1.14269707]


## 7 Se considera el sistema no lineal

$$
3x^2-y^2=0
$$
$$
3xy^2-x^3-1=0
$$

a. Obtener la matriz jacobiana asociada al sistema no lineal de ecuaciones.
Sera mostrada en el anexo consiguiente despues del codigo

In [9]:

# 1. DEFINIMOS EL SISTEMA Y EL JACOBIANO (Ejercicio 7)
def F7(w):
    x, y = w
    return np.array([3*x**2 - y**2, 3*x*y**2 - x**3 - 1])

def JF7(w):
    x, y = w
    return np.array([
        [6*x, -2*y],
        [3*y**2 - 3*x**2, 6*x*y]
    ])

# 2. DEFINIMOS LOS TRES MÉTODOS
def NewtonSis(F, JF, x0, tol, maxiter):
    iters = 0; error = tol + 1; x0 = np.array(x0, dtype=float)
    while error > tol and iters <= maxiter:
        z = np.linalg.solve(JF(x0), F(x0))
        x = x0 - z
        error = np.linalg.norm(x - x0)
        x0 = x; iters += 1
    return [x, iters, error]

def TrapeciosSis(F, JF, x0, tol, maxiter):
    iters = 0; error = tol + 1; x0 = np.array(x0, dtype=float)
    while error > tol and iters <= maxiter:
        z = np.linalg.solve(JF(x0), F(x0))
        y = x0 - z
        # Fórmula de Trapecios
        M = 2 * (JF(y) + JF(x0))
        v = np.linalg.solve(M, F(x0))
        x = x0 - v
        error = np.linalg.norm(x - x0)
        x0 = x; iters += 1
    return [x, iters, error]

def SimpsonSis(F, JF, x0, tol, maxiter):
    iters = 0; error = tol + 1; x0 = np.array(x0, dtype=float)
    while error > tol and iters <= maxiter:
        z = np.linalg.solve(JF(x0), F(x0))
        y = x0 - z
        mid = (x0 + y) / 2
        # Fórmula de Simpson
        M = 6 * (JF(x0) + 4*JF(mid) + JF(y))
        v = np.linalg.solve(M, F(x0))
        x = x0 - v
        error = np.linalg.norm(x - x0)
        x0 = x; iters += 1
    return [x, iters, error]

# 3. EJECUTAMOS LOS TRES MÉTODOS
x0 = [1.0, 1.0]
tol = 1e-6
maxiter = 30

sol_n, it_n, err_n = NewtonSis(F7, JF7, x0, tol, maxiter)
sol_t, it_t, err_t = TrapeciosSis(F7, JF7, x0, tol, maxiter)
sol_s, it_s, err_s = SimpsonSis(F7, JF7, x0, tol, maxiter)


In [10]:
# 4. CREAMOS LA TABLA COMPARATIVA
tabla = pd.DataFrame({
    "Método": ["Newton", "Trapecios", "Simpson"],
    "Solución [x, y]": [sol_n, sol_t, sol_s],
    "Iteraciones": [it_n, it_t, it_s],
    "Error": [err_n, err_t, err_s]
})

# Mostramos la tabla
print("--- COMPARACIÓN DE MÉTODOS (EJERCICIO 7) ---")
display(tabla)

--- COMPARACIÓN DE MÉTODOS (EJERCICIO 7) ---


,Método,"Solución [x, y]",Iteraciones,Error
0,Newton,"[0.5, 0.8660254037844387]",5,1.967492e-09
1,Trapecios,"[0.5000691813505369, 0.8660375571782446]",31,2.341359e-05
2,Simpson,"[0.7134099841392416, 0.9136238379425817]",31,6.223191e-03
